## 1. Environment Setup and Data Loading
In this section, we will import required Python libraries for data processing, visualization and machine learning. Then, we will load the insurance dataset and examine its structure to understand the features we will be working with.

In [ ]:
# Libraries for data manipulation
import pandas as pd
import numpy as np

# Libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Libraries for machine learning and model evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
# loading the dataset
df = pd.read_csv("/kaggle/input/datasets/awaiskaggler/insurance-csv/insurance.csv")

# View the first 5 rows of the dataset to see what it looks
display(df.head())

# Print a concise summary of the DataFrame (column names, non-null counts, and data types)
print("\n--- Dataset Info ---")
df.info()

# Display basic statistical details like percentile, mean, standard deviation, etc.
print("\n--- Statistical Summary ---")
df.describe()

## 2. Data Preparation: Handling Categorical Variables

Machine learning algorithms need numerical inputs. The dataset contains categorical features (`sex`, `smoker`, `region`) that need to be converted into numerical format. To achieve this, we will use **One-Hot Encoding** to prevent any unintended ordinal relationships between categories. To avoid multicollinearity (Dummy Variable Trap) we will drop the first dummy variable for each original category.

In [ ]:
# Create dummy variables for categorical columns
# drop_first=True prevents the dummy variable trap by removing the first coded column
df_converted = pd.get_dummies(df, columns=["sex", "smoker", "region"], drop_first=True)

# If the get_dummies function returns Boolean values, convert the Boolean values (True/False) to integers (1/0)
df_converted = df_converted.astype(int)

# Display the new structure of the dataset
print("--- Dataset after One-Hot Encoding---")
display(df_converted.head())

# Check the new shape of the dataframe
print(f"\nNew shape of the dataset: {df_converted.shape}")

## 3. Train-Test Split and Linear Regression Baseline

Before going to Polynomial Regression, we create a simple Linear Regression model. This gives us a baseline to test whether adding more complexity to our model (adding polynomial terms) actually improves our predictions. We will split our data into `training` and `testing` sets, 80% and 20% respectively, to evaluate the model on unseen data.

In [ ]:
# Separate features (X) and target variable (y)
X = df_converted.drop("expenses", axis=1)
y = df_converted["expenses"]

# Split the data into training (80%) and testing (20%) sets.
# random_states=42 is a seed value used to ensure that the code divides with the same randomness each time it is run.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Start and train the basic linear regression model
lin_reg_model = LinearRegression()
lin_reg_model.fit(X_train, y_train)

# Prediction on the test data
y_pred_lin = lin_reg_model.predict(X_test)

## 4. Evaluating the Baseline Linear Regression Model

To understand how well our basic linear model performs, we will calculate three key metrics:

- **MAE (Mean Absolute Error):** The average absolute difference between predicted and actual expenses.

- **MSE / RMSE (Mean Squared Error / Root Mean Squared Error):** MSE is the average square difference of the values. Because the values are squared, small errors become unimportant, while large errors (outliers) become significant. RMSE converts the MSE result back into a readable value by taking its square root.

- **R² Score:** It shows what percentage of the variation in the actual data the model can explain.

In [ ]:
# Calculate evaluation metrics for the linear regression model
mae_lin = mean_absolute_error(y_test, y_pred_lin)
mse_lin = mean_squared_error(y_test, y_pred_lin)
rmse_lin = np.sqrt(mse_lin) # Square root of MSE
r2_lin = r2_score(y_test, y_pred_lin)

# The results formatted to 2 decimal places
print("--- Baseline Linear Regression Performance ---")
print(f"MAE      : ${mae_lin:,.2f}")
print(f"MSE      : {mse_lin:,.2f}")
print(f"RMSE     : ${rmse_lin:,.2f}")
print(f"R2 Score : {r2_lin:.4f}")

## 5. Hyperparameter Tuning: Finding the Optimal Polynomial Degree

We will use `PolynomialFeatures` from scikit-learn to capture the non-linear relationships in our data. This function creates a new feature matrix consisting of all polynomial combinations of features. After transforming our features, we train a standard `Linear Regression` model on this new, expanded dataset. 

But first, we need to find the optimal polynomial degree to prevent overfitting. We will loop through degrees 1 to 5, train a new model for each, and record the Root Mean Squared Error (RMSE) for both the training and testing datasets. Plotting these errors will visually reveal where the model stops generalizing and starts memorizing the training data.

In [ ]:
# Lists to store the RMSE results for different degrees
train_rmse_errors = []
test_rmse_errors = []

# Loop from degree 1 to 5 (inclusive)
degrees = range(1, 6)

for d in degrees:
    # Create polynomial features for the current degree
    poly_converter = PolynomialFeatures(degree=d, include_bias=False)
    X_train_poly = poly_converter.fit_transform(X_train)
    X_test_poly = poly_converter.transform(X_test)

    # Train the model
    model = LinearRegression()
    model.fit(X_train_poly, y_train)

    # Predict on both train and test data
    train_preds = model.predict(X_train_poly)
    test_preds = model.predict(X_test_poly)

    # Calculate RMSE for both
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))

    # Append the results to our lists
    train_rmse_errors.append(train_rmse)
    test_rmse_errors.append(test_rmse)

# --- Plotting the Results ---
plt.figure(figsize=(8, 4))
plt.plot(degrees, train_rmse_errors, label="Train RMSE", color="blue", marker="o")
plt.plot(degrees, test_rmse_errors, label="Test RMSE", color="red", marker="s")

plt.title('Train vs Test RMSE for Different Polynomial Degrees')
plt.xlabel('Polynomial Degree')
plt.ylabel('RMSE (Dollars)')
plt.legend()
plt.grid(True)
plt.show()

Our best degree is where the RMSE in the test data is lowest and then rises. As shown in the graph, `2` is the most suitable degree for us. 

## 6. Polynomial Regression Model

In [ ]:
# Create polynomial features
poly_converter = PolynomialFeatures(degree=2, include_bias=False)

# Transform the training and testing data
# We use fit_transform on training data to learn the parameters and transform it
X_train_poly = poly_converter.fit_transform(X_train)
# We ONLY use transform on testing data to apply the same rules (no fitting to prevent data leakage)
X_test_poly = poly_converter.transform(X_test)

print(f"Original feature count: {X_train.shape[1]}")
print(f"Polynomial feature count: {X_train_poly.shape[1]}\n")

# Train a new Linear Regression model on the polynomial features
poly_reg_model = LinearRegression()
poly_reg_model.fit(X_train_poly, y_train)

# Make predictions using the transformed test data
y_pred_poly = poly_reg_model.predict(X_test_poly)

# Evaluate the new model
mae_poly = mean_absolute_error(y_test, y_pred_poly)
mse_poly = mean_squared_error(y_test, y_pred_poly)
rmse_poly = np.sqrt(mse_poly)
r2_poly = r2_score(y_test, y_pred_poly)

print("--- Polynomial Regression (Degree=2) Performance ---")
print(f"MAE      : ${mae_poly:,.2f}")
print(f"MSE      : {mse_poly:,.2f}")
print(f"RMSE     : ${rmse_poly:,.2f}")
print(f"R2 Score : {r2_poly:.4f}")


## 7. Final Evaluation: Actual vs. Predicted Expenses

Since we have multiple features, we cannot plot a simple 2D polynomial curve. The standard approach for evaluating multivariate regression models visually is an **Actual vs. Predicted** scatter plot. We will plot our best model's predictions (degree=2) against the actual test values. A perfect model would have all points aligned exactly on the diagonal red dashed line (y = x).

In [ ]:
# Create the Actual vs. Predicted scatter plot
plt.figure(figsize=(8, 8))
# Plot the actual vs predicted points
plt.scatter(y_test, y_pred_poly, alpha=0.6, color='teal', label='Predictions')

#  Draw the ideal 45-degree line (Perfect Prediction Line)
max_val = max(y_test.max(), y_pred_poly.max())
plt.plot([0, max_val], [0, max_val], color='red', linestyle='--', linewidth=2, label='Perfect Fit (y=x)')

plt.title("Actual vs. Predicted Insurance Charges (Degree=2)")
plt.xlabel("Actual Expenses ($)")
plt.ylabel("Predicted Expenses ($)")
plt.legend()
plt.grid(True)
plt.show()